## Healthcare Domain Example
Developing a Clinical Decision Support System (CDSS) for diagnosing congenital heart disease in newborns.
The example is based on the [CHILD network](https://www.bnlearn.com/bnrepository/discrete-medium.html) which is originally a Bayesian network. 

### 0. Import functions shared with the other domain examples

In [1]:
import sys
sys.path.append("..")
from yarrrmlHandler import YarrrmlHandler
from etc import export_sheets_to_csv
from rdf_etc import run_query_file, add_namespaces, show_rdf_graph

### 1. Export all Excel worksheets as csv files (YARRRML does not inherently support xlsx files)

In [2]:
for filename in ['claimGenerationNetwork', 'claimFormalizationNetwork']:
    export_sheets_to_csv(
        excel_file=filename + '.xlsx', 
        output_dir='csv'
)

Exported experts -> csv\experts.csv
Exported events -> csv\events.csv
Exported resources -> csv\resources.csv
Exported claims -> csv\claims.csv
Exported claims -> csv\claims.csv
Exported kEngineers -> csv\kEngineers.csv
Exported formalRepresentations -> csv\formalRepresentations.csv
Exported assessments -> csv\assessments.csv
Exported CQs -> csv\CQs.csv
Exported formalCQs -> csv\formalCQs.csv


### 2. Convert csv data to RDF representation using the YARRRML specification
The graph is visualized using [yfiles-graphs-for-jupter](https://www.yworks.com/products/yfiles-graphs-for-jupyter). Please refer to the official documentation for using the exploration features. The visualization is designed for networkx graphs, not RDF graphs in specific. If you want to explore the graph with a specialized tool, please export the graph by uncommenting the serialization contextmanager.

In [4]:
from rdflib import Graph
from rdf_etc import convert_and_targets, add_inferred_provenance

try:
    result = YarrrmlHandler.run_pipeline(
        working_dir=".",
        yarrrml_file="healthcare.yarrrml.yml",
        rml_file="healthcare.rml.ttl",
    )
    print("Pipeline executed successfully.")
except Exception as ex:
    print(f"Error: {ex}")

g =  Graph().parse(data=result.stdout, format="nt")
g = add_namespaces(g)
g = convert_and_targets(g)
g = add_inferred_provenance(g)

# Uncomment to export RDF graph
# with open("rdf_export.ttl", mode="w", encoding="utf8") as f:
#     f.write(g.serialize(format="turtle"))


show_rdf_graph(g)

Pipeline executed successfully.


GraphWidget(layout=Layout(height='800px', width='100%'))

### 3. Store conditional knowledge base as annotation page as snapshot for further processing

In [32]:
from etc import output_pretty_rdf

# Choose expert between A and F; transitive "no" returns only the knowledge of the chosen expert, "yes" returns all experts as all experts know each other (in)directly.
result = run_query_file(g, "get_experts_template.rq", {"expert": "ex:C", "transitive": "yes"})
experts = {f"<{str(row[0])}>" for row in result}

result = run_query_file(g, "get_network_kb_template.rq", {"experts": ", ". join(experts)})
constructed_graph = Graph()
for triple in result:
    constructed_graph.add(triple)

constructed_graph = add_namespaces(constructed_graph)
g += constructed_graph
ttl = constructed_graph.serialize(format="turtle")

# Output is shortend due to its length
output_pretty_rdf(ttl[:300])

### 4. Extract applicable rules from the knowledge base and save as ProbLog file

In [34]:
import pandas as pd

results =  run_query_file(g, "extract_rules_template.rq", {"knowledge_base":"ex:NetworkKnowledgeBase"})

df = pd.DataFrame(
    [(str(row.item), str(row.bodyValue)) for row in results],
    columns=["Node", "Rule"]
)

display(df)

with open("kb.problog", mode="w", encoding="utf8") as f:
    f.write("\n".join(df.Rule.tolist()))    

,Node,Rule
0,http://www.example.com/F1,0.1::birthasphyxia(yes).
1,http://www.example.com/F10,0.23469388::disease(paivs) :- birthasphyxia(no).
2,http://www.example.com/F100,0.75::lungflow(high) :- disease(tga).
3,http://www.example.com/F101,0.05::lungflow(high) :- disease(fallot).
4,http://www.example.com/F102,0.05::lungflow(high) :- disease(paivs).
...,...,...
321,http://www.example.com/F95,0.8::lungflow(low) :- disease(fallot).
322,http://www.example.com/F96,0.85::lungflow(low) :- disease(paivs).
323,http://www.example.com/F97,0.1::lungflow(low) :- disease(tapvd).
324,http://www.example.com/F98,0.1::lungflow(low) :- disease(lung).


### 5. Generate the CDSS based on the exported knowledge base
* Select variables of interest in the _Query tab_. The system will return the probabilities of all variable values.
* Determine any observed evidence in the _Evidence tab_.
* Press "Run Inference" to start the reasoning process.

In [35]:
from functools import partial
from IPython.display import display
import ipywidgets as widgets

from gui import create_ui, run_inference
from problog_etc import get_variable_domains_from_problog

PROBLOG_FILE = "kb.problog" # "If desired, use "child.problog" as reference. It is a direct translation from the Bayesian network
var_domains = get_variable_domains_from_problog(PROBLOG_FILE)
ui, spinner, run_button, evidence_widgets, query_widgets =  create_ui(var_domains)

output = widgets.Output()
run_button.on_click(partial(run_inference, 
                            problog_file=PROBLOG_FILE, 
                            var_domains=var_domains, 
                            spinner=spinner,
                            output=output, 
                            evidence_widgets=evidence_widgets,
                            query_widgets=query_widgets))

display(ui)
display(output)

Output()